In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit

In [2]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CONFIG = json.loads((ROOT / 'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [3]:
from sklearn.linear_model import LogisticRegression

In [4]:
TAG = 'ar6_6'
ORDER = 6

In [5]:
pairs = pd.read_csv(INPUT / 'transitions.csv', dtype={'stage':str})
panel = pd.read_csv(INPUT / 'panel.csv', index_col='dblp_id')
q0 = panel['0'].dropna().to_numpy()
DEST = OUTPUT / TAG
DEST.mkdir(exist_ok=True)

In [6]:
# lag1 = q_t, lag6 = q_(t-5); complete observed windows only
Q = panel.to_numpy(dtype=float)
blocks, outcomes, ages = [], [], []
for t in range(ORDER-1, Y):
    valid = np.isfinite(Q[:,t-ORDER+1:t+2]).all(axis=1)
    blocks.append(np.log1p(Q[valid,t-ORDER+1:t+1])[:,::-1])
    outcomes.append(Q[valid,t+1])
    ages.append(np.repeat(t, valid.sum()))
X, target = np.vstack(blocks), np.concatenate(outcomes)
activity = LogisticRegression(C=1000, max_iter=1500, random_state=SEED)
activity.fit(X, (target>0).astype(int))
positive = target>0
Xp = np.column_stack([np.ones(positive.sum()), X[positive]])
coef = np.linalg.lstsq(Xp, np.log(target[positive]), rcond=None)[0]
sigma = np.sqrt(np.mean((np.log(target[positive])-Xp@coef)**2))
params = pd.DataFrame({'lag':np.arange(1,ORDER+1), 'activity_beta':activity.coef_[0], 'positive_beta':coef[1:],
                       'activity_intercept':activity.intercept_[0], 'positive_intercept':coef[0],
                       'positive_sigma':sigma, 'n_activity':len(target), 'n_positive':positive.sum()})
params.to_csv(DEST / 'parameters.csv', index=False)
pd.DataFrame({'age':np.concatenate(ages)}).value_counts().sort_index().rename('n').to_csv(DEST / 'fit_support.csv')
display(params)

,lag,activity_beta,positive_beta,activity_intercept,positive_intercept,positive_sigma,n_activity,n_positive
0,1,0.733246,0.318221,-1.105612,0.186857,0.598807,16945,14872
1,2,0.583091,0.222676,-1.105612,0.186857,0.598807,16945,14872
2,3,0.379082,0.116426,-1.105612,0.186857,0.598807,16945,14872
3,4,0.170497,0.077275,-1.105612,0.186857,0.598807,16945,14872
4,5,0.192224,0.041385,-1.105612,0.186857,0.598807,16945,14872
5,6,0.204217,0.041798,-1.105612,0.186857,0.598807,16945,14872
